# 🎭 Project 01: Sentiment Analyzer

**What you'll learn:**
- What sentiment analysis is and why it matters
- How to use two popular NLP tools: **VADER** and **TextBlob**
- How to analyze text sentiment without training any model yourself

**Time to complete:** ~20 minutes

---

## 🧠 What is Sentiment Analysis?

Sentiment analysis (also called *opinion mining*) is the process of using AI to determine whether a piece of text expresses a **positive**, **negative**, or **neutral** emotion.

**Real-world uses:**
- Analyzing customer reviews on Amazon, Yelp, etc.
- Monitoring brand reputation on Twitter/X
- Understanding feedback in surveys
- Stock market sentiment from news headlines

---

## Step 1 — Install Dependencies

Run this once to install all required libraries.

In [ ]:
# Run this cell once to install dependencies
import sys
!{sys.executable} -m pip install nltk textblob pandas matplotlib --quiet

## Step 2 — Import Libraries

In [ ]:
import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from textblob import TextBlob
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib

# Download VADER's pre-trained word dictionary
nltk.download('vader_lexicon', quiet=True)

print('✅ All libraries imported successfully!')

## Step 3 — Understanding VADER

**VADER** stands for *Valence Aware Dictionary and sEntiment Reasoner*. It was specifically designed for **social media text** — it understands things like:
- Capitalization: `AMAZING` is more positive than `amazing`
- Punctuation: `great!!!` is more positive than `great`
- Emojis and slang

VADER gives a **compound score** from **-1** (very negative) to **+1** (very positive).

In [ ]:
# Create a VADER analyzer
sia = SentimentIntensityAnalyzer()

# Analyze a sentence
text = "I absolutely LOVE this product!! It's amazing!"
scores = sia.polarity_scores(text)

print(f'Text: "{text}"')
print(f'\nVADER Scores:')
print(f'  Positive : {scores["pos"]:.3f}  (fraction of text that is positive)')
print(f'  Negative : {scores["neg"]:.3f}  (fraction of text that is negative)')
print(f'  Neutral  : {scores["neu"]:.3f}  (fraction of text that is neutral)')
print(f'  Compound : {scores["compound"]:+.3f}  (overall sentiment: -1 to +1)')

# Interpret the compound score
if scores['compound'] >= 0.05:
    print(f'\n→ Overall Sentiment: 😊 POSITIVE')
elif scores['compound'] <= -0.05:
    print(f'\n→ Overall Sentiment: 😞 NEGATIVE')
else:
    print(f'\n→ Overall Sentiment: 😐 NEUTRAL')

## Step 4 — Understanding TextBlob

**TextBlob** gives two scores:
- **Polarity**: -1.0 (very negative) to +1.0 (very positive)
- **Subjectivity**: 0.0 (very objective/factual) to 1.0 (very subjective/opinionated)

Subjectivity is useful: a sentence like *"The sky is blue"* is objective (0.0), while *"This is the best day ever!"* is very subjective (close to 1.0).

In [ ]:
texts_to_compare = [
    "The sky is blue.",                                  # objective fact
    "I think the sky looks beautiful today.",            # subjective opinion
    "This is the worst movie I have ever seen!",         # negative & subjective
    "The package was delivered in 3 days.",              # neutral fact
]

print(f'{'Text':<45} {'Polarity':>10} {'Subjectivity':>14} {'Label':>10}')
print('-' * 85)

for text in texts_to_compare:
    blob = TextBlob(text)
    polarity = blob.sentiment.polarity
    subjectivity = blob.sentiment.subjectivity
    
    if polarity > 0:
        label = 'Positive'
    elif polarity < 0:
        label = 'Negative'
    else:
        label = 'Neutral'
    
    short_text = text[:44]
    print(f'{short_text:<45} {polarity:>+10.3f} {subjectivity:>14.3f} {label:>10}')

## Step 5 — Analyzing a Batch of Reviews

In [ ]:
# Sample customer reviews
reviews = [
    "Absolutely fantastic product! Best purchase I've made all year.",
    "Terrible quality. Broke after one day. Total waste of money.",
    "It arrived on time. The color matches the picture.",
    "Super happy with this!! Five stars all the way!!",
    "Not great, not terrible. It's okay for the price.",
    "Horrible customer service. Never buying from this brand again.",
    "Decent product. Does what it's supposed to do.",
    "WOW! Exceeded all my expectations. Highly recommend!",
]

# Analyze all reviews
results = []
for review in reviews:
    vader_score = sia.polarity_scores(review)['compound']
    blob_score = TextBlob(review).sentiment.polarity
    
    vader_label = 'Positive' if vader_score >= 0.05 else ('Negative' if vader_score <= -0.05 else 'Neutral')
    blob_label = 'Positive' if blob_score > 0 else ('Negative' if blob_score < 0 else 'Neutral')
    
    results.append({
        'Review': review[:55] + '...',
        'VADER': vader_label,
        'VADER Score': f'{vader_score:+.3f}',
        'TextBlob': blob_label,
        'TextBlob Score': f'{blob_score:+.3f}',
    })

df = pd.DataFrame(results)
print(df.to_string(index=False))

## Step 6 — Visualize the Results

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Sentiment Analysis Results', fontsize=15, fontweight='bold')

colors = {'Positive': '#4CAF50', 'Neutral': '#2196F3', 'Negative': '#F44336'}

for ax, col, title in zip(axes, ['VADER', 'TextBlob'], ['VADER', 'TextBlob']):
    counts = df[col].value_counts()
    bar_colors = [colors.get(l, 'gray') for l in counts.index]
    bars = ax.bar(counts.index, counts.values, color=bar_colors, width=0.5, edgecolor='white', linewidth=1.5)
    ax.set_title(f'{title} Sentiment Distribution', fontsize=12)
    ax.set_ylabel('Number of Reviews')
    ax.set_ylim(0, len(reviews))
    for bar, val in zip(bars, counts.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                str(val), ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.savefig('sentiment_distribution.png', dpi=120, bbox_inches='tight')
plt.show()
print('Chart saved!')

## Step 7 — Try It Yourself!

Edit the `my_texts` list below and run the cell to analyze your own sentences.

In [ ]:
# ✏️ Edit these texts and re-run the cell!
my_texts = [
    "I love learning about artificial intelligence!",
    "This notebook is confusing and hard to follow.",
    "The weather is okay today.",
]

print(f'{'Your Text':<50} {'VADER':>10} {'TextBlob':>10}')
print('-' * 75)
for text in my_texts:
    v = sia.polarity_scores(text)['compound']
    t = TextBlob(text).sentiment.polarity
    v_label = 'Positive' if v >= 0.05 else ('Negative' if v <= -0.05 else 'Neutral')
    t_label = 'Positive' if t > 0 else ('Negative' if t < 0 else 'Neutral')
    print(f'{text[:50]:<50} {v_label:>10} {t_label:>10}')

## 🏁 Summary

| | VADER | TextBlob |
|--|-------|----------|
| **Best for** | Social media, short text | General text |
| **Extra insight** | Raw pos/neg/neu breakdown | Subjectivity score |
| **Speed** | Very fast | Fast |
| **Training required** | No | No |

**Next steps:**
- Try analyzing real tweets or product reviews
- Explore `transformers` library for state-of-the-art sentiment models (BERT)
- Move on to **Project 02 — House Price Predictor** to learn supervised learning!